In [1]:
# =========================================================
# INSTALL REQUIRED LIBRARIES
# =========================================================

!pip install -q flask
!pip install -q pyngrok
!pip install -q transformers
!pip install -q streamlit

# =========================================================
# IMPORT LIBRARIES
# =========================================================

import numpy as np
import pandas as pd
import os
import re
import torch

from flask import Flask, request, jsonify

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline
)

from pyngrok import ngrok

# =========================================================
# CLEAR GPU MEMORY
# =========================================================

torch.cuda.empty_cache()

# =========================================================
# DISPLAY INPUT FILES
# =========================================================

for dirname, _, filenames in os.walk('/kaggle/input'):

    for filename in filenames:

        print(os.path.join(dirname, filename))

# =========================================================
# INITIALIZE FLASK APPLICATION
# =========================================================

app = Flask(__name__)

# =========================================================
# MODEL NAME
# =========================================================

model_name = "Srr1234/tinyllama-qlora-chatbot"

# =========================================================
# LOAD MODEL AND TOKENIZER
# =========================================================

def load_model_and_tokenizer(model_name):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(model_name)

    return model, tokenizer

# =========================================================
# LOAD MODEL
# =========================================================

model, tokenizer = load_model_and_tokenizer(model_name)

print("MODEL LOADED SUCCESSFULLY!")

# =========================================================
# TEXT GENERATION FUNCTION
# =========================================================

def generate_text(prompt, max_length=100):

    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_length=max_length
    )

    result = pipe(
        f"<s>[INST] {prompt} [/INST]",
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
    )[0]['generated_text']

    # REMOVE PROMPT TEMPLATE
    result = re.sub(
        r"<s>\[INST\].*?\[/INST\]",
        "",
        result
    ).strip()

    return result

# =========================================================
# HOME ROUTE
# =========================================================

@app.route('/')

def home():

    return "TinyLlama Chatbot API is Running Successfully!"

# =========================================================
# GENERATE ROUTE
# =========================================================

@app.route('/generate', methods=['POST'])

def generate():

    data = request.json

    prompt = data.get('prompt')

    response_text = generate_text(prompt)

    return jsonify({
        'generated_text': response_text
    })

# =========================================================
# RUN FLASK + NGROK
# =========================================================

if __name__ == '__main__':

    try:

        # =================================================
        # ADD YOUR NGROK AUTH TOKEN
        # =================================================

        ngrok.set_auth_token("3DVALNsdmhLDWOFnzUDXRHUYIsi_5J6Vb6iy2eLK8V39PhE9M")

        # =================================================
        # CREATE PUBLIC URL
        # =================================================

        http_tunnel = ngrok.connect(
            addr='5000',
            proto='http'
        )

        print(f"\nPUBLIC URL:\n{http_tunnel.public_url}")

        print(f"\nGENERATE ENDPOINT:\n{http_tunnel.public_url}/generate")

        # =================================================
        # RUN FLASK APP
        # =================================================

        app.run(
            host='0.0.0.0',
            port=5000
        )

    except Exception as e:

        print(f"\nERROR:\n{str(e)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 68.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

MODEL LOADED SUCCESSFULLY!

PUBLIC URL:
https://imagines-levitate-scenic.ngrok-free.dev

GENERATE ENDPOINT:
https://imagines-levitate-scenic.ngrok-free.dev/generate
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [10/May/2026 06:53:24] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/May/2026 06:53:25] "GET /favicon.ico HTTP/1.1" 404 -
Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'top_p', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
INFO:werkzeug:127.0.0.1 - 

#Test this prompt

import requests

url = "https://imagines-levitate-scenic.ngrok-free.dev/generate"

data = {
    "prompt": "What is artificial intelligence?"
}

response = requests.post(url, json=data)

print(response.json())